# S3 — Pandas 轉換：groupby / merge / pivot（解答版）

> **對應**：`S3_pandas_transform.ipynb` 課堂練習  
> **資料**：`orders_clean.csv` + `customers.csv` + `products.csv`  
> **說明**：本 notebook 只包含「課堂練習」的解答（送分題／核心題／挑戰題），並加上教學提示，方便學員對照檢查。

---

## 解題策略

- **送分題**：單表 `groupby + agg`，先確認資料欄位。
- **核心題**：先用 `merge` 合併三表，再針對題目做分組聚合。
- **挑戰題 (RFM)**：用 `groupby.agg` 的 **named aggregation** 一次產出 R/F/M 三個欄位，再 `merge` 顧客名稱。


In [ ]:
import pandas as pd
import numpy as np

DATA = '../datasets/ecommerce'
orders    = pd.read_csv(f'{DATA}/orders_clean.csv', parse_dates=['order_date'])
customers = pd.read_csv(f'{DATA}/customers.csv')
products  = pd.read_csv(f'{DATA}/products.csv')

# 分析基底：三表合併（核心題與挑戰題會用到）
df = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(products,  on='product_id',  how='left')
)
print('orders   :', orders.shape)
print('df (三表):', df.shape)

---
## 🟢 送分題 — 解答

1. 每個 `customer_id` 的訂單總數
2. 平均單筆訂單金額

**教學提示**：兩題都只需要 `orders` 單表。第 1 題用 `size()` 或 `count()`；第 2 題用 `mean()`。一次 `agg` 同時算完更漂亮。


In [ ]:
# 解法 A：分開算
order_count = orders.groupby('customer_id').size()
mean_amount = orders.groupby('customer_id')['amount'].mean()

print('每位顧客訂單數 (head):')
print(order_count.head())
print('\n每位顧客平均金額 (head):')
print(mean_amount.head().round(2))

In [ ]:
# 解法 B（推薦）：一次 agg 兩個指標，可讀性高
easy = (
    orders.groupby('customer_id')
          .agg(訂單數=('order_id', 'count'),
               平均金額=('amount', 'mean'))
          .round(2)
          .reset_index()
)
easy.head()

---
## 🟡 核心題 — 解答

用三表合併的 `df` 回答：
1. 哪個**商品類別**總營收最高？
2. **Gold** 等級的顧客一共下了多少筆訂單？金額多少？
3. 各**地區**的平均訂單金額？

**教學提示**：
- Q1：`groupby('category')['amount'].sum()` 後 `sort_values` 或 `idxmax()` 抓最高者。
- Q2：先 `df[df['vip_level']=='Gold']` 篩選，再對 `amount` 同時算 `count + sum`。
- Q3：`groupby('region')['amount'].mean()`。


In [ ]:
# Q1：商品類別總營收排名
category_rev = (
    df.groupby('category')['amount']
      .sum()
      .sort_values(ascending=False)
)
print('🏆 類別營收排名:')
print(category_rev)
print(f'\n→ 營收最高類別：{category_rev.idxmax()}（{category_rev.max():,.0f}）')

In [ ]:
# Q2：Gold 等級顧客的訂單數與總金額
gold = df[df['vip_level'] == 'Gold']
gold_stat = gold['amount'].agg(['count', 'sum'])
print('💎 Gold 等級貢獻:')
print(f"  訂單數：{int(gold_stat['count'])}")
print(f"  總金額：{gold_stat['sum']:,.0f}")

In [ ]:
# Q3：各地區的平均訂單金額
region_mean = (
    df.groupby('region')['amount']
      .mean()
      .round(2)
      .sort_values(ascending=False)
)
print('📍 各地區平均訂單金額:')
print(region_mean)

---
## 🔴 挑戰題 — RFM 解答

計算每位顧客的 **RFM**：
- **R**ecency：最近一次下單日期（`max(order_date)`）
- **F**requency：下單次數
- **M**onetary：總金額

最後用 M 排序，取 Top 5 並附上顧客名字。

**教學提示**：
1. **Recency 的實務做法**：真實專案中 Recency 通常是「距離今天（或 snapshot date，常設為資料集 `max(order_date) + 1 day`）的天數」，數值越小代表越近期。本題為了教學簡化，直接用 `max(order_date)` 當 R 欄位即可，後面註解區塊示範如何轉成天數版本。
2. **named aggregation**：`groupby.agg(R=('order_date','max'), F=('order_id','count'), M=('amount','sum'))` 一次產出三欄，語意最清楚。
3. **join 顧客名字**：用 `merge(customers[['customer_id','customer_name']], on='customer_id', how='left')` 把名字貼上來。


In [ ]:
# Step 1：用 named aggregation 一次算出 R / F / M
rfm = (
    orders.groupby('customer_id')
          .agg(
              R=('order_date', 'max'),
              F=('order_id',   'count'),
              M=('amount',     'sum'),
          )
          .reset_index()
)
print('RFM 形狀:', rfm.shape)
rfm.head()

In [ ]:
# Step 2：merge 顧客名字
rfm_named = rfm.merge(
    customers[['customer_id', 'customer_name']],
    on='customer_id',
    how='left',
)

# Step 3：用 M 排序，取 Top 5
top5 = (
    rfm_named
    .sort_values('M', ascending=False)
    .head(5)
    .reset_index(drop=True)
    [['customer_id', 'customer_name', 'R', 'F', 'M']]
)
print('🏆 RFM Top 5 顧客（依 Monetary 排序）:')
top5

In [ ]:
# Bonus：把 Recency 轉成「距離 snapshot date 的天數」
# 實務常見做法：snapshot = max(order_date) + 1 day，避免最近一單的 Recency = 0
snapshot = orders['order_date'].max() + pd.Timedelta(days=1)
print('snapshot date:', snapshot.date())

rfm_days = rfm_named.copy()
rfm_days['R_days'] = (snapshot - rfm_days['R']).dt.days
rfm_days = rfm_days[['customer_id', 'customer_name', 'R_days', 'F', 'M']]
rfm_days.sort_values('M', ascending=False).head(5)

---
## 重點回顧

| 題型 | 核心技巧 |
|---|---|
| 送分題 | `groupby + agg` 一次多指標 |
| 核心題 | 先 `merge` 再 `groupby`；`idxmax()` 抓冠軍 |
| RFM | named aggregation + `merge` 補欄位；snapshot date 轉天數 |

**做完這張 notebook 你應該能回答**：
- 為什麼 RFM 要用 snapshot date，而不是直接用今天？  
  → 確保分析結果可重現（reproducible），不會隨著「今天」改變；同時避免最近一筆訂單 R=0 造成的除零/分箱問題。
- `groupby.agg(新欄=('原欄','func'))` 比傳 dict 好在哪？  
  → 語意更清楚、欄名直接命名、不會產生 MultiIndex，後續處理更簡單。
